In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np

BASE_PATH = "/content/drive/MyDrive/Fraud_Project"

print("Project path exists:", os.path.exists(BASE_PATH))

Project path exists: True


In [ ]:
final_dataset_path = os.path.join(
    BASE_PATH,
    "final_financial_mda_dataset.csv"
)

df = pd.read_csv(final_dataset_path)

print("Dataset shape:", df.shape)
print("\nFraud distribution:")
print(df["is_fraud"].value_counts())

print("\nDuplicate CIK-year pairs:",
      df.duplicated(subset=["cik", "fiscal_year"]).sum())

print("\nUnique companies:",
      df["cik"].nunique())

print("\nFraud company-years:",
      (df["is_fraud"] == 1).sum())

print("\nNon-fraud company-years:",
      (df["is_fraud"] == 0).sum())

Dataset shape: (31241, 23)

Fraud distribution:
is_fraud
0    30965
1      276
Name: count, dtype: int64

Duplicate CIK-year pairs: 0

Unique companies: 2931

Fraud company-years: 276

Non-fraud company-years: 30965


In [ ]:
split_files = {
    "Train": "E1_train.csv",
    "Validation": "E1_validation.csv",
    "Test": "E1_test.csv"
}

split_data = {}

for split_name, filename in split_files.items():
    path = os.path.join(BASE_PATH, filename)
    split_data[split_name] = pd.read_csv(path)

    print(f"{split_name}:")
    print("  Rows:", len(split_data[split_name]))
    print("  Companies:", split_data[split_name]["cik"].nunique())
    print("  Fraud:", (split_data[split_name]["is_fraud"] == 1).sum())
    print("  Non-fraud:", (split_data[split_name]["is_fraud"] == 0).sum())
    print()

train_df = split_data["Train"]
val_df = split_data["Validation"]
test_df = split_data["Test"]

print("Company overlap checks:")
print("Train ∩ Validation:",
      len(set(train_df["cik"]) & set(val_df["cik"])))
print("Train ∩ Test:",
      len(set(train_df["cik"]) & set(test_df["cik"])))
print("Validation ∩ Test:",
      len(set(val_df["cik"]) & set(test_df["cik"])))

Train:
  Rows: 18734
  Companies: 1761
  Fraud: 155
  Non-fraud: 18579

Validation:
  Rows: 6322
  Companies: 586
  Fraud: 57
  Non-fraud: 6265

Test:
  Rows: 6185
  Companies: 584
  Fraud: 64
  Non-fraud: 6121

Company overlap checks:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [ ]:
results_path = os.path.join(
    BASE_PATH,
    "Comparative_Analysis_Results.csv"
)

verified_results = pd.read_csv(results_path)

print("Number of experiment results:", len(verified_results))
print("\nExperiments:")
print(verified_results["Experiment"].value_counts().sort_index())

print("\nFull results table:")
display(
    verified_results.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "ROC_AUC": "{:.4f}",
        "PR_AUC": "{:.4f}"
    })
)


Number of experiment results: 7

Experiments:
Experiment
E1    3
E2    2
E3    2
Name: count, dtype: int64

Full results table:


,Experiment,Model,Precision,Recall,F1,ROC_AUC,PR_AUC
0,E1,XGBoost,0.0175,0.1875,0.0321,0.5220,0.0137
1,E1,LightGBM,0.0139,0.0781,0.0236,0.5264,0.0118
2,E1,MLP,0.0286,0.2344,0.0510,0.5588,0.0269
3,E2,TF-IDF + Logistic Regression,0.0309,0.3438,0.0566,0.7300,0.0321
4,E2,TF-IDF + MLP,0.0317,0.0625,0.0421,0.6491,0.0198
5,E3,Hybrid Logistic Regression,0.0426,0.1250,0.0635,0.5209,0.0161
6,E3,Hybrid MLP,0.0258,0.1875,0.0454,0.6840,0.0252


In [ ]:
artifact_files = [
    "E1_XGBoost_model.pkl",
    "E1_LightGBM_model.pkl",
    "E1_MLP_model.pkl",
    "E1_MLP_imputer.pkl",
    "E1_MLP_scaler.pkl",
    "E1_financial_imputer.pkl",
    "E2_TFIDF_vectorizer.pkl",
    "E2_TFIDF_LogisticRegression_model.pkl",
    "E2_TFIDF_threshold.txt",
    "E2_TFIDF_MLP_model.pkl",
    "E2_TFIDF_MLP_threshold.txt",
    "E3_Hybrid_MLP_model.pkl",
    "E3_Hybrid_MLP_SVD.pkl",
    "E3_Hybrid_MLP_scaler.pkl",
    "E3_Hybrid_MLP_threshold.txt"
]

artifact_check = pd.DataFrame({
    "Artifact": artifact_files,
    "Exists": [
        os.path.exists(os.path.join(BASE_PATH, f))
        for f in artifact_files
    ]
})

display(artifact_check)

print(
    "\nAll required artifacts exist:",
    artifact_check["Exists"].all()
)

,Artifact,Exists
0,E1_XGBoost_model.pkl,True
1,E1_LightGBM_model.pkl,True
2,E1_MLP_model.pkl,True
3,E1_MLP_imputer.pkl,True
4,E1_MLP_scaler.pkl,True
5,E1_financial_imputer.pkl,True
6,E2_TFIDF_vectorizer.pkl,True
7,E2_TFIDF_LogisticRegression_model.pkl,True
8,E2_TFIDF_threshold.txt,True
9,E2_TFIDF_MLP_model.pkl,True



All required artifacts exist: True


In [ ]:
threshold_files = {
    "E1 XGBoost": None,
    "E1 LightGBM": None,
    "E1 MLP": None,
    "E2 TF-IDF + LR": "E2_TFIDF_threshold.txt",
    "E2 TF-IDF + MLP": "E2_TFIDF_MLP_threshold.txt",
    "E3 Hybrid LR": None,
    "E3 Hybrid MLP": "E3_Hybrid_MLP_threshold.txt"
}

print("Saved thresholds:\n")

for model_name, filename in threshold_files.items():
    if filename is not None:
        path = os.path.join(BASE_PATH, filename)
        with open(path, "r") as f:
            value = f.read().strip()

        print(f"{model_name}: {value}")

print("\nThresholds used in final evaluation:")
print("E1 XGBoost: 0.05")
print("E1 LightGBM: 0.0005")
print("E1 MLP: 0.02")
print("E2 TF-IDF + LR: 0.10")
print("E2 TF-IDF + MLP: 0.002")
print("E3 Hybrid LR: 0.70")
print("E3 Hybrid MLP: 0.01")

Saved thresholds:

E2 TF-IDF + LR: 0.1
E2 TF-IDF + MLP: 0.002
E3 Hybrid MLP: 0.01

Thresholds used in final evaluation:
E1 XGBoost: 0.05
E1 LightGBM: 0.0005
E1 MLP: 0.02
E2 TF-IDF + LR: 0.10
E2 TF-IDF + MLP: 0.002
E3 Hybrid LR: 0.70
E3 Hybrid MLP: 0.01


In [ ]:
shap_files = [
    "E3_Hybrid_MLP_SHAP_Summary.png",
    "E3_Hybrid_MLP_SHAP_feature_importance.csv",
    "E3_Hybrid_MLP_SHAP_financial_features.csv",
    "E3_Hybrid_MLP_SHAP_text_features.csv",
    "E3_Hybrid_MLP_SHAP_representative_fraud_case.csv",
    "E3_Hybrid_MLP_SHAP_representative_clean.csv"
]

shap_check = pd.DataFrame({
    "SHAP Output": shap_files,
    "Exists": [
        os.path.exists(os.path.join(BASE_PATH, f))
        for f in shap_files
    ]
})

display(shap_check)

print(
    "\nAll SHAP outputs exist:",
    shap_check["Exists"].all()
)

,SHAP Output,Exists
0,E3_Hybrid_MLP_SHAP_Summary.png,True
1,E3_Hybrid_MLP_SHAP_feature_importance.csv,True
2,E3_Hybrid_MLP_SHAP_financial_features.csv,True
3,E3_Hybrid_MLP_SHAP_text_features.csv,True
4,E3_Hybrid_MLP_SHAP_representative_fraud_case.csv,True
5,E3_Hybrid_MLP_SHAP_representative_clean.csv,True



All SHAP outputs exist: True


In [ ]:
shap_importance_path = os.path.join(
    BASE_PATH,
    "E3_Hybrid_MLP_SHAP_feature_importance.csv"
)

shap_importance = pd.read_csv(shap_importance_path)

financial_features = [
    "price",
    "Assets",
    "AssetsCurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "CommonStockSharesIssued",
    "CommonStockValue",
    "IncomeTaxExpenseBenefit",
    "LiabilitiesCurrent",
    "OperatingIncomeLoss",
    "StockholdersEquity",
    "NetIncomeLoss",
    "Liabilities",
    "current_ratio",
    "cash_ratio",
    "current_assets_ratio",
    "liabilities_to_assets",
    "debt_to_equity",
    "roa",
    "roe",
    "operating_return_on_assets"
]

shap_importance["feature_group"] = np.where(
    shap_importance["feature"].isin(financial_features),
    "Financial",
    "MD&A / SVD"
)

group_summary = (
    shap_importance
    .groupby("feature_group")["mean_abs_shap"]
    .sum()
    .reset_index()
)

total_shap = group_summary["mean_abs_shap"].sum()

group_summary["percentage"] = (
    group_summary["mean_abs_shap"] / total_shap * 100
)

display(
    group_summary.style.format({
        "mean_abs_shap": "{:.6f}",
        "percentage": "{:.2f}%"
    })
)

,feature_group,mean_abs_shap,percentage
0,Financial,0.002307,11.91%
1,MD&A / SVD,0.017070,88.09%
